# VQA_trainB_1_translation.ipynb
Tạo translation.json cho translation bridge VI→EN.

In [ ]:
#Cell_0 - Kaggle-safe imports for translation bridge
# Notebook này tạo translation.json: question_vi/answer_vi -> question_en/answer_en.
# Không upgrade transformers/torch để tránh xung đột Kaggle; chỉ cài sentencepiece nếu thiếu.
import sys, subprocess, importlib.util, os, json, gc, hashlib, warnings
from pathlib import Path
from typing import List, Dict, Any

def ensure_pkg(pkg, pip_name=None):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name or pkg])

ensure_pkg('sentencepiece', 'sentencepiece')

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

warnings.filterwarnings('ignore')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)


In [ ]:
!ls /kaggle/input/datasets/khangnhut/vqa-vietnameses-food-dataset

In [ ]:
#Cell_1 - Config paths
CANDIDATE_DATA_DIRS = [
    Path('/kaggle/input/datasets/khangnhut/vqa-vietnameses-food-dataset'),
]
DATA_DIR = next((p for p in CANDIDATE_DATA_DIRS if (p/'train.json').exists()), CANDIDATE_DATA_DIRS[0])
OUT_DIR = Path('/kaggle/working'); OUT_DIR.mkdir(parents=True, exist_ok=True)
TRANSLATION_JSON = OUT_DIR / 'translation.json'
CACHE_JSON = OUT_DIR / 'translation_cache.json'

VI_EN_MODEL = 'Helsinki-NLP/opus-mt-vi-en'
EN_VI_MODEL = 'Helsinki-NLP/opus-mt-en-vi'
BATCH_SIZE_TRANSLATE = 16
MAX_LEN = 96
SPLITS = ['train', 'val', 'test']
# Debug nhanh: đặt MAX_ROWS_PER_SPLIT = 200. Train thật: None.
MAX_ROWS_PER_SPLIT = None
print('DATA_DIR:', DATA_DIR)
print('OUT_DIR:', OUT_DIR)


In [ ]:
#Cell_2 - Load split data and normalize image paths
def load_rows(path):
    obj = json.loads(Path(path).read_text(encoding='utf-8'))
    rows = obj if isinstance(obj, list) else obj.get('data', [])
    if MAX_ROWS_PER_SPLIT is not None:
        rows = rows[:MAX_ROWS_PER_SPLIT]
    return rows

def resolve_relative_image_path(raw_path, image_id=None):
    # Giữ relative path trong translation.json để portable; TrainB_2 sẽ resolve bằng DATA_DIR.
    p = Path(str(raw_path))
    if p.is_absolute():
        return str(Path('vietnamese_food_images_2k5/vietnamese_food_images_2k5') / p.name)
    parts = list(p.parts)
    if len(parts) >= 2 and parts[-2] in {'vietnamese_food_images_2k5/vietnamese_food_images_2k5', 'vietnamese_food_images'}:
        return str(Path(parts[-2]) / parts[-1])
    if p.name:
        return str(Path('vietnamese_food_images_2k5/vietnamese_food_images_2k5') / p.name)
    return str(Path('vietnamese_food_images_2k5/vietnamese_food_images_2k5') / f'image_{image_id}.jpg')

all_rows = []
for split in SPLITS:
    rows = load_rows(DATA_DIR / f'{split}.json')
    for i, r in enumerate(rows):
        item = dict(r)
        item['split'] = split
        item['row_id'] = f"{split}_{i}_{item.get('sample_id', item.get('image_id', ''))}"
        item['image_path'] = resolve_relative_image_path(item.get('image_path', ''), item.get('image_id'))
        all_rows.append(item)
print('Loaded rows:', len(all_rows), {s: sum(r['split']==s for r in all_rows) for s in SPLITS})


In [ ]:
#Cell_3 - Translation utilities with cache
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.ipc_collect()

def load_translator(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(DEVICE).eval()
    return tok, mdl

def translate_texts(texts: List[str], tok, mdl, batch_size=16, max_len=96):
    outs = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc='translate'):
            batch = [str(x) for x in texts[i:i+batch_size]]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(DEVICE)
            gen = mdl.generate(**enc, max_new_tokens=max_len, num_beams=3)
            outs.extend(tok.batch_decode(gen, skip_special_tokens=True))
    return outs

def cache_key(direction, text):
    return hashlib.md5((direction + '||' + str(text)).encode('utf-8')).hexdigest()

cache = {}
if CACHE_JSON.exists():
    cache = json.loads(CACHE_JSON.read_text(encoding='utf-8'))
print('Cache size:', len(cache))

def translate_unique(texts, direction, model_name):
    missing = []
    for t in texts:
        k = cache_key(direction, t)
        if k not in cache:
            missing.append(t)
    missing = list(dict.fromkeys(missing))
    print(direction, 'unique missing:', len(missing))
    if missing:
        tok, mdl = load_translator(model_name)
        translated = translate_texts(missing, tok, mdl, BATCH_SIZE_TRANSLATE, MAX_LEN)
        for src, tgt in zip(missing, translated):
            cache[cache_key(direction, src)] = tgt
        CACHE_JSON.write_text(json.dumps(cache, ensure_ascii=False, indent=2), encoding='utf-8')
        del tok, mdl
        clear_memory()
    return [cache[cache_key(direction, t)] for t in texts]


In [ ]:
#Cell_4 - Translate VI -> EN for questions and full answers
# B1 chỉ làm nhiệm vụ dịch:
# - question_vi -> question_en để phục vụ B1 zero-shot và B2 fine-tuning.
# - raw_answer/answer -> answer_vi -> answer_en để làm ground truth cho B2 fine-tuning/evaluation.
# Không rút gọn câu trả lời ở bước này.
def pick_full_answer_vi(row):
    raw = str(row.get('raw_answer', '')).strip()
    if raw:
        return raw
    return str(row.get('answer', '')).strip()

questions_vi = [str(r.get('question', '')).strip() for r in all_rows]
answers_vi = [pick_full_answer_vi(r) for r in all_rows]
questions_en = translate_unique(questions_vi, 'vi-en', VI_EN_MODEL)
answers_en = translate_unique(answers_vi, 'vi-en', VI_EN_MODEL)

translated_rows = []
for r, q_vi, a_vi, q_en, a_en in zip(all_rows, questions_vi, answers_vi, questions_en, answers_en):
    item = dict(r)
    # Giữ raw_answer gốc trong item để trace/debug, nhưng ground truth chính của B pipeline là answer_vi/answer_en.
    item['question_vi'] = q_vi
    item['answer_vi'] = a_vi
    item['question_en'] = q_en
    item['answer_en'] = a_en
    item['answer_source'] = 'raw_answer' if str(r.get('raw_answer', '')).strip() else 'answer'
    translated_rows.append(item)
print('Translated rows:', len(translated_rows))
print('Answer source counts:', {k: sum(r.get('answer_source') == k for r in translated_rows) for k in ['raw_answer', 'answer']})
print('Sample:', {k: translated_rows[0].get(k) for k in ['question_vi','question_en','answer_vi','answer_en','answer_source']})


In [ ]:
#Cell_5 - Save translation.json and per-split files
TRANSLATION_JSON.write_text(json.dumps(translated_rows, ensure_ascii=False, indent=2), encoding='utf-8')
for split in SPLITS:
    rows = [r for r in translated_rows if r['split'] == split]
    (OUT_DIR / f'translation_{split}.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
config = {
    'data_dir': str(DATA_DIR),
    'out_dir': str(OUT_DIR),
    'translation_json': str(TRANSLATION_JSON),
    'vi_en_model': VI_EN_MODEL,
    'en_vi_model': EN_VI_MODEL,
    'counts': {s: sum(r['split']==s for r in translated_rows) for s in SPLITS},
}
(OUT_DIR / 'translation_config.json').write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(config, ensure_ascii=False, indent=2))
